# DATASCAPE 2026 — Rusak Specialist + Label Refinement, ConvNeXt-Base 384

Notebook tahap kedua setelah training ConvNeXt-Base label asli. Tujuannya memperbaiki pemisahan **Aman–Rusak** tanpa mengorbankan kelas **Lainnya**.

Pipeline ini tidak membiarkan satu model mengubah labelnya sendiri:

1. memuat probabilitas OOF dan checkpoint lima fold dari model utama;
2. melatih binary specialist Aman–Rusak secara OOF dengan dua view saat training dan enam view saat inferensi;
3. menggabungkan probabilitas specialist dengan model tiga kelas melalui kalibrasi cross-fold;
4. memakai hasil gabungan hanya bila macro-F1 dan F1 Rusak naik, sementara F1 Aman/Lainnya tidak turun material;
5. membuat kandidat relabel hanya ketika model utama, specialist, dan teacher lama sepakat kuat;
6. label baru baru diterapkan bila ada keputusan manual pada CSV review.

Aktifkan **A100 GPU**, pastikan folder hasil training Base sebelumnya masih berada di Drive, kemudian pilih **Run all**.

In [ ]:
# Kaggle API bootstrap (configured for this project)
import os
from pathlib import Path

KAGGLE_API_TOKEN = 'KGAT_c2a604d54464b1bc7271fb95428085f2'
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN

kaggle_directory = Path.home() / '.kaggle'
kaggle_directory.mkdir(parents=True, exist_ok=True)
kaggle_access_token = kaggle_directory / 'access_token'
kaggle_access_token.write_text(KAGGLE_API_TOKEN + '\n')
kaggle_access_token.chmod(0o600)
print('Kaggle API token configured:', kaggle_access_token)


## Batas penggunaan label

- Semua prediksi untuk baris train dibuat out-of-fold.
- Kandidat model tidak otomatis dianggap ground truth.
- Tanpa file review manual, `label_refined` tetap sama dengan label resmi.
- Baris berstatus `unclear` dapat diberi bobot lebih kecil pada training berikutnya; kandidat yang belum diperiksa tetap berbobot penuh.
- Test set hanya dipakai setelah seluruh parameter penggabungan dipilih dari OOF.

In [ ]:
import importlib.util
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

required = {
    'numpy': 'numpy', 'pandas': 'pandas', 'PIL': 'Pillow',
    'matplotlib': 'matplotlib', 'seaborn': 'seaborn',
    'sklearn': 'scikit-learn', 'torch': 'torch',
    'timm': 'timm', 'huggingface_hub': 'huggingface-hub',
    'tqdm': 'tqdm',
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import timm
import torch
import torch.nn.functional as F
from IPython.display import display
from PIL import Image, ImageDraw, ImageOps
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_recall_fscore_support
from timm.utils import ModelEmaV3
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm

IN_COLAB = 'google.colab' in sys.modules
print('Colab:', IN_COLAB, '| PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 1. Konfigurasi dan sumber model utama

Specialist dimulai dari bobot setiap fold model utama, kemudian head tiga kelas diganti menjadi binary head. Stem dan dua stage awal dibekukan agar training lebih cepat dan tidak merusak fitur umum.

In [ ]:
SEED = 42
CLASSES = ('Aman', 'Rusak', 'Lainnya')
CLASS_TO_INDEX = {label: index for index, label in enumerate(CLASSES)}
COMPETITION = 'datascape-2026-data-mining-competition'
MODEL_ID = 'hf_hub:timm/convnext_base.fb_in22k_ft_in1k_384'

USE_GOOGLE_DRIVE = True
SOURCE_RUN_DIR = Path('/content/drive/MyDrive/DATASCAPE/final_convnext_base_original_384_amp_5fold_6epoch_a100_eb256')
OUTPUT_DIR = Path('/content/drive/MyDrive/DATASCAPE/rusak_specialist_refinement_384_a100')
LOCAL_OUTPUT_DIR = Path('artifacts/rusak_specialist_refinement_384_a100')
LOCAL_DATA_PATH = ''
MANIFEST_FILENAME = 'student_training_manifest.csv'
MANIFEST_SEARCH_ROOT = Path('/content/drive/MyDrive/DATASCAPE')
MANUAL_REVIEW_CSV = ''  # Opsional: CSV hasil review dengan filepath, manual_decision, manual_label.

IMAGE_SIZE = 384
TRAIN_VIEWS = 2
EVAL_VIEWS = 6
FOLDS = 5
MAX_EPOCHS = 4
PATIENCE = 2
REFERENCE_LR = 5e-5
MIN_LR = 2e-6
WEIGHT_DECAY = 3e-2
FOCAL_GAMMA = 1.5
MODEL_EMA_DECAY = 0.9995
TARGET_EFFECTIVE_BATCH_SIZE = 128
BATCH_CANDIDATES = (32, 24, 16, 12, 8, 4)
NUM_WORKERS = 4
GRAD_CLIP_NORM = 1.0
FREEZE_EARLY_STAGES = 2
RESUME_FOLDS = True

# Primary hanya menerima specialist bila aturan ini lolos pada cross-fitted OOF.
MIN_MACRO_GAIN = 0.001
MIN_RUSAK_F1_GAIN = 0.005
MAX_PROTECTED_CLASS_DROP = 0.003

# Kandidat relabel harus disetujui tiga sinyal OOF dan tetap perlu review manusia.
MIN_MAIN_CONFIDENCE = 0.90
MIN_SPECIALIST_CONFIDENCE = 0.90
MIN_LEGACY_CONFIDENCE = 0.90

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    OUTPUT_DIR = LOCAL_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')
print('Output:', OUTPUT_DIR)

In [ ]:
def get_kaggle_token():
    token = os.environ.get('KAGGLE_API_TOKEN', '').strip()
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = (userdata.get('KAGGLE_API_TOKEN') or token).strip()
        except Exception:
            pass
    if not token:
        raise RuntimeError('Set KAGGLE_API_TOKEN di Colab Secrets atau environment.')
    return token


def safe_extract(zip_path, destination):
    zip_path, destination = Path(zip_path), Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(root):
                raise ValueError(f'Unsafe ZIP member: {member.filename}')
        archive.extractall(destination)
    return destination


def find_dataset_root(start):
    start = Path(start).expanduser().resolve()
    candidates = [start]
    if start.is_dir():
        candidates.extend(path.parent for path in start.rglob('sample_submission.csv'))
    for root in candidates:
        if (root / 'sample_submission.csv').is_file() and (root / 'Test_Set').is_dir():
            if all((root / 'Train_Set' / label).is_dir() for label in CLASSES):
                return root
    raise FileNotFoundError(f'Root dataset tidak ditemukan di {start}')


def resolve_dataset_root():
    candidates = [Path(LOCAL_DATA_PATH).expanduser()] if LOCAL_DATA_PATH else []
    candidates += [Path.cwd() / COMPETITION, Path('/content') / COMPETITION]
    for candidate in candidates:
        if candidate.exists():
            try:
                return find_dataset_root(candidate)
            except FileNotFoundError:
                pass
    if importlib.util.find_spec('kagglehub') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
    import kagglehub
    from kagglehub.config import set_kaggle_api_token
    set_kaggle_api_token(get_kaggle_token())
    downloaded = Path(kagglehub.competition_download(COMPETITION))
    if downloaded.is_file() and downloaded.suffix.lower() == '.zip':
        return find_dataset_root(safe_extract(downloaded, Path('/content/datascape_dataset')))
    return find_dataset_root(downloaded)


def resolve_manifest_path():
    direct_candidates = [
        SOURCE_RUN_DIR / MANIFEST_FILENAME,
        MANIFEST_SEARCH_ROOT / MANIFEST_FILENAME,
        Path('/content') / MANIFEST_FILENAME,
        Path.cwd() / MANIFEST_FILENAME,
    ]
    for path in direct_candidates:
        if path.is_file():
            return path
    if MANIFEST_SEARCH_ROOT.is_dir():
        matches = sorted(MANIFEST_SEARCH_ROOT.glob(f'*/{MANIFEST_FILENAME}'))
        if matches:
            return matches[0]
    if IN_COLAB:
        from google.colab import files
        print('Pilih student_training_manifest.csv dari run preprocessing.')
        uploaded = files.upload()
        if MANIFEST_FILENAME in uploaded:
            path = Path('/content') / MANIFEST_FILENAME
            path.write_bytes(uploaded[MANIFEST_FILENAME])
            return path
    raise FileNotFoundError(MANIFEST_FILENAME)


required_source_files = ['oof_probabilities.npy', 'test_probabilities_raw.npy']
missing_source = [name for name in required_source_files if not (SOURCE_RUN_DIR / name).is_file()]
missing_checkpoints = [fold for fold in range(FOLDS) if not (SOURCE_RUN_DIR / f'fold_{fold}' / 'best.pt').is_file()]
if missing_source or missing_checkpoints:
    raise FileNotFoundError(
        f'Run utama tidak lengkap di {SOURCE_RUN_DIR}. '
        f'File hilang={missing_source}, checkpoint fold hilang={missing_checkpoints}'
    )

DATA_ROOT = resolve_dataset_root()
MANIFEST_PATH = resolve_manifest_path()
print('Dataset:', DATA_ROOT)
print('Manifest:', MANIFEST_PATH)
print('Main run:', SOURCE_RUN_DIR)

## 2. Validasi alignment data dan artefak OOF

Jumlah dan urutan baris harus identik dengan run utama. Notebook berhenti bila alignment tidak dapat dibuktikan.

In [ ]:
train_frame = pd.read_csv(MANIFEST_PATH).reset_index(drop=True)
required_columns = {
    'filepath', 'label_original', 'fold', 'split_group', 'pixel_sha256',
    'oof_prob_Aman', 'oof_prob_Rusak', 'oof_prob_Lainnya',
}
missing = required_columns - set(train_frame.columns)
if missing:
    raise ValueError(f'Kolom manifest tidak lengkap: {sorted(missing)}')
if not set(train_frame.label_original).issubset(CLASSES):
    raise ValueError('Label manifest tidak sesuai.')
if train_frame.groupby('split_group').fold.nunique().max() != 1:
    raise ValueError('Split group menyeberangi fold.')

train_frame['image_path'] = train_frame.filepath.map(lambda value: str(DATA_ROOT / Path(value)))
missing_paths = train_frame.loc[~train_frame.image_path.map(lambda value: Path(value).is_file()), 'filepath']
if len(missing_paths):
    raise FileNotFoundError(f'{len(missing_paths)} gambar train tidak ditemukan; contoh: {missing_paths.iloc[0]}')

template = pd.read_csv(DATA_ROOT / 'sample_submission.csv', dtype={'filename': str})
test_root = DATA_ROOT / 'Test_Set'
test_by_name = {path.name: path for path in test_root.rglob('*') if path.is_file()}
missing_test = [name for name in template.filename if name not in test_by_name]
if missing_test:
    raise FileNotFoundError(f'{len(missing_test)} gambar test tidak ditemukan; contoh: {missing_test[0]}')
test_frame = pd.DataFrame({
    'filename': template.filename,
    'image_path': [str(test_by_name[name]) for name in template.filename],
})

main_oof = np.load(SOURCE_RUN_DIR / 'oof_probabilities.npy').astype(np.float32)
main_test = np.load(SOURCE_RUN_DIR / 'test_probabilities_raw.npy').astype(np.float32)
if main_oof.shape != (len(train_frame), len(CLASSES)):
    raise ValueError(f'OOF shape {main_oof.shape} tidak cocok dengan manifest {len(train_frame)}.')
if main_test.shape != (len(test_frame), len(CLASSES)):
    raise ValueError(f'Test probability shape {main_test.shape} tidak cocok dengan template {len(test_frame)}.')
if not np.allclose(main_oof.sum(1), 1.0, atol=1e-4) or not np.allclose(main_test.sum(1), 1.0, atol=1e-4):
    raise ValueError('Probabilitas model utama tidak ternormalisasi.')

targets = train_frame.label_original.map(CLASS_TO_INDEX).to_numpy()
legacy_binary = train_frame.oof_prob_Rusak.to_numpy() / np.clip(
    train_frame.oof_prob_Aman.to_numpy() + train_frame.oof_prob_Rusak.to_numpy(), 1e-7, None
)
print('Train:', len(train_frame), '| Test:', len(test_frame))
display(pd.crosstab(train_frame.fold, train_frame.label_original).reindex(columns=CLASSES, fill_value=0))

## 3. Multi-view dataset

Saat training, specialist melihat gambar global dan satu crop acak. Saat validasi/test, specialist melihat gambar global, crop tengah, dan empat crop sudut. Crop dibuat dari resolusi asli sebelum resize sehingga cacat kecil tidak langsung hilang akibat letterbox.

In [ ]:
def load_rgb(path):
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image)
        if image.mode == 'P':
            image = image.convert('RGBA')
        if image.mode in {'RGBA', 'LA'} or 'transparency' in image.info:
            rgba = image.convert('RGBA')
            background = Image.new('RGBA', rgba.size, (124, 116, 104, 255))
            image = Image.alpha_composite(background, rgba)
        return image.convert('RGB').copy()


def letterbox(image):
    fill = tuple(int(round(value * 255)) for value in IMAGENET_MEAN)
    return ImageOps.pad(image, (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR, color=fill)


def image_to_uint8(image):
    array = np.asarray(image, dtype=np.uint8).copy()
    return torch.from_numpy(array).permute(2, 0, 1)


def square_crop(image, center_x, center_y, scale):
    width, height = image.size
    side = max(8, int(round(min(width, height) * scale)))
    left = int(round(center_x * width - side / 2))
    top = int(round(center_y * height - side / 2))
    left = min(max(left, 0), max(width - side, 0))
    top = min(max(top, 0), max(height - side, 0))
    return image.crop((left, top, left + side, top + side)).resize(
        (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR
    )


def train_views(image):
    scale = random.uniform(0.58, 0.88)
    half = scale / 2
    center_x = random.uniform(half, 1 - half)
    center_y = random.uniform(half, 1 - half)
    return [letterbox(image), square_crop(image, center_x, center_y, scale)]


def eval_views(image):
    scale = 0.70
    half = scale / 2
    positions = [
        (0.5, 0.5),
        (half, half), (1 - half, half),
        (half, 1 - half), (1 - half, 1 - half),
    ]
    return [letterbox(image)] + [square_crop(image, x, y, scale) for x, y in positions]


class MultiViewDataset(Dataset):
    def __init__(self, frame, training=False, labeled=True):
        self.frame = frame.reset_index(drop=True)
        self.training = training
        self.labeled = labeled
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = load_rgb(row.image_path)
        views = train_views(image) if self.training else eval_views(image)
        tensor = torch.stack([image_to_uint8(view) for view in views])
        if self.labeled:
            target = float(row.label_original == 'Rusak')
            return tensor, target, index
        return tensor, index


def balanced_binary_sampler(frame):
    counts = frame.label_original.value_counts()
    weights = frame.label_original.map(lambda label: 0.5 / counts[label]).to_numpy()
    return WeightedRandomSampler(torch.tensor(weights, dtype=torch.double), len(frame), replacement=True)


def make_loader(dataset, batch_size, sampler=None, shuffle=False):
    options = dict(
        dataset=dataset, batch_size=batch_size, sampler=sampler,
        shuffle=shuffle if sampler is None else False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )
    if NUM_WORKERS > 0:
        options.update(persistent_workers=True, prefetch_factor=2)
    return DataLoader(**options)


preview = MultiViewDataset(train_frame.head(2), training=False)
print('Eval views:', tuple(preview[0][0].shape))

## 4. Specialist model dan binary focal loss

Pooling memberi bobot pada global view sekaligus crop dengan indikasi Rusak terkuat. Untuk gambar Aman, false positive pada salah satu crop tetap mendapat penalti dari focal loss.

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('Aktifkan GPU Colab.')

GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
COMPUTE_CAPABILITY = torch.cuda.get_device_capability(0)
USE_BF16 = COMPUTE_CAPABILITY[0] >= 8 and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
USE_SCALER = AMP_DTYPE == torch.float16
MEAN = torch.tensor(IMAGENET_MEAN, device=DEVICE).view(1, 1, 3, 1, 1)
STD = torch.tensor(IMAGENET_STD, device=DEVICE).view(1, 1, 3, 1, 1)


def build_specialist(main_checkpoint):
    model = timm.create_model(MODEL_ID, pretrained=False, num_classes=len(CLASSES), drop_rate=0.15, drop_path_rate=0.15)
    checkpoint = torch.load(main_checkpoint, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model_state'])
    model.reset_classifier(1)
    if hasattr(model, 'stem'):
        for parameter in model.stem.parameters():
            parameter.requires_grad = False
    if hasattr(model, 'stages'):
        for stage in model.stages[:FREEZE_EARLY_STAGES]:
            for parameter in stage.parameters():
                parameter.requires_grad = False
    return model.to(DEVICE, memory_format=torch.channels_last)


def prepare_views(images, training):
    images = images.to(DEVICE, dtype=torch.float32, non_blocking=True).div_(255.0)
    batch, views = images.shape[:2]
    if training:
        flip = torch.rand(batch, device=DEVICE) < 0.5
        if flip.any():
            images[flip] = torch.flip(images[flip], dims=[-1])
        brightness = torch.empty(batch, 1, 1, 1, 1, device=DEVICE).uniform_(0.96, 1.04)
        images = (images * brightness).clamp_(0, 1)
    images = (images - MEAN) / STD
    return images.reshape(batch * views, 3, IMAGE_SIZE, IMAGE_SIZE).contiguous(memory_format=torch.channels_last), batch, views


def view_logits(model, images, training):
    flat, batch, views = prepare_views(images, training)
    logits = model(flat).reshape(batch, views)
    return logits


def pool_specialist_logits(logits):
    global_logit = logits[:, 0]
    if logits.shape[1] == 1:
        return global_logit
    strongest_local = logits[:, 1:].amax(dim=1)
    return 0.55 * global_logit + 0.45 * strongest_local


def binary_focal_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    probability = torch.sigmoid(logits)
    p_t = probability * targets + (1 - probability) * (1 - targets)
    return (((1 - p_t) ** FOCAL_GAMMA) * bce).mean()


def probe_batch_size():
    checkpoint = SOURCE_RUN_DIR / 'fold_0' / 'best.pt'
    for batch_size in BATCH_CANDIDATES:
        model = None
        try:
            torch.cuda.empty_cache()
            model = build_specialist(checkpoint)
            optimizer = torch.optim.AdamW(
                [p for p in model.parameters() if p.requires_grad], lr=REFERENCE_LR
            )
            dummy = torch.zeros((batch_size, TRAIN_VIEWS, 3, IMAGE_SIZE, IMAGE_SIZE), dtype=torch.uint8)
            targets_probe = torch.zeros(batch_size, device=DEVICE)
            with torch.autocast('cuda', dtype=AMP_DTYPE):
                loss = binary_focal_loss(pool_specialist_logits(view_logits(model, dummy, True)), targets_probe)
            loss.backward(); optimizer.step()
            del dummy, targets_probe, loss, optimizer, model
            torch.cuda.empty_cache()
            return batch_size
        except (torch.cuda.OutOfMemoryError, RuntimeError) as error:
            message = str(error).lower()
            if 'out of memory' not in message and 'engine' not in message and 'cudnn' not in message:
                raise
            if model is not None:
                del model
            torch.cuda.empty_cache()
    raise RuntimeError('Tidak ada batch size yang muat.')


BATCH_SIZE = probe_batch_size()
GRAD_ACCUM_STEPS = max(1, math.ceil(TARGET_EFFECTIVE_BATCH_SIZE / BATCH_SIZE))
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUM_STEPS
EVAL_BATCH_SIZE = max(4, min(24, BATCH_SIZE))
BASE_LR = REFERENCE_LR * EFFECTIVE_BATCH_SIZE / TARGET_EFFECTIVE_BATCH_SIZE
print(
    'GPU:', GPU_NAME, '| AMP:', str(AMP_DTYPE).replace('torch.', ''),
    '| physical batch:', BATCH_SIZE, '| effective batch:', EFFECTIVE_BATCH_SIZE,
    '| eval batch:', EVAL_BATCH_SIZE,
)

In [ ]:
def learning_rate_at(epoch):
    if MAX_EPOCHS <= 1:
        return BASE_LR
    progress = epoch / (MAX_EPOCHS - 1)
    return MIN_LR + (BASE_LR - MIN_LR) * 0.5 * (1 + math.cos(math.pi * progress))


def predict_specialist(model, loader):
    model.eval()
    aggregate_all, global_all, maximum_all, minimum_all, indices_all = [], [], [], [], []
    with torch.inference_mode():
        for batch in tqdm(loader, desc='specialist predict', leave=False):
            if len(batch) == 3:
                images, _, indices = batch
            else:
                images, indices = batch
            with torch.autocast('cuda', dtype=AMP_DTYPE):
                logits = view_logits(model, images, False)
                aggregate = torch.sigmoid(pool_specialist_logits(logits))
                per_view = torch.sigmoid(logits)
            aggregate_all.append(aggregate.float().cpu().numpy())
            global_all.append(per_view[:, 0].float().cpu().numpy())
            maximum_all.append(per_view.max(1).values.float().cpu().numpy())
            minimum_all.append(per_view.min(1).values.float().cpu().numpy())
            indices_all.append(indices.numpy())
    order = np.argsort(np.concatenate(indices_all))
    return {
        'probability': np.concatenate(aggregate_all)[order],
        'global_probability': np.concatenate(global_all)[order],
        'view_max': np.concatenate(maximum_all)[order],
        'view_min': np.concatenate(minimum_all)[order],
    }


def binary_metrics(frame, probabilities):
    mask = frame.label_original.isin(['Aman', 'Rusak']).to_numpy()
    truth = frame.loc[mask, 'label_original'].eq('Rusak').astype(int).to_numpy()
    predicted = (probabilities[mask] >= 0.5).astype(int)
    return f1_score(truth, predicted, average='macro', zero_division=0)


def train_specialist_fold(fold):
    output_fold = OUTPUT_DIR / f'fold_{fold}'
    output_fold.mkdir(parents=True, exist_ok=True)
    complete = output_fold / 'complete.json'
    checkpoint_path = output_fold / 'specialist_best.pt'
    array_paths = {
        'valid': output_fold / 'valid_predictions.npz',
        'test': output_fold / 'test_predictions.npz',
    }
    if RESUME_FOLDS and complete.is_file() and all(path.is_file() for path in array_paths.values()):
        return (
            dict(np.load(array_paths['valid'])),
            dict(np.load(array_paths['test'])),
            json.loads(complete.read_text()),
        )

    train = train_frame[
        train_frame.fold.ne(fold) & train_frame.label_original.isin(['Aman', 'Rusak'])
    ].reset_index(drop=True)
    valid_all = train_frame[train_frame.fold.eq(fold)].reset_index(drop=True)
    valid_binary = valid_all[valid_all.label_original.isin(['Aman', 'Rusak'])].reset_index(drop=True)
    train_loader = make_loader(
        MultiViewDataset(train, training=True), BATCH_SIZE,
        sampler=balanced_binary_sampler(train),
    )
    valid_binary_loader = make_loader(MultiViewDataset(valid_binary, training=False), EVAL_BATCH_SIZE)

    model = build_specialist(SOURCE_RUN_DIR / f'fold_{fold}' / 'best.pt')
    ema = ModelEmaV3(model, decay=MODEL_EMA_DECAY, use_warmup=True, warmup_gamma=1.0, warmup_power=2 / 3)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=BASE_LR, weight_decay=WEIGHT_DECAY,
    )
    scaler = torch.amp.GradScaler('cuda', enabled=USE_SCALER)
    best_score, stale, updates, history = -1.0, 0, 0, []

    for epoch in range(MAX_EPOCHS):
        lr = learning_rate_at(epoch)
        for group in optimizer.param_groups:
            group['lr'] = lr
        model.train(); optimizer.zero_grad(set_to_none=True)
        total_loss, total, started = 0.0, 0, time.time()
        number_of_batches = len(train_loader)
        for step, (images, targets_binary, _) in enumerate(tqdm(train_loader, desc=f'fold {fold} epoch {epoch + 1}', leave=False)):
            targets_binary = targets_binary.to(DEVICE, dtype=torch.float32, non_blocking=True)
            window_start = (step // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
            window_size = min(GRAD_ACCUM_STEPS, number_of_batches - window_start)
            with torch.autocast('cuda', dtype=AMP_DTYPE):
                logits = pool_specialist_logits(view_logits(model, images, True))
                raw_loss = binary_focal_loss(logits, targets_binary)
                loss = raw_loss / window_size
            scaler.scale(loss).backward()
            should_step = (step + 1) % GRAD_ACCUM_STEPS == 0 or step + 1 == number_of_batches
            if should_step:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
                updates += 1; ema.update(model, step=updates)
            total_loss += float(raw_loss.item()) * len(images); total += len(images)

        valid_result = predict_specialist(ema.module, valid_binary_loader)
        score = binary_metrics(valid_binary, valid_result['probability'])
        row = {'epoch': epoch + 1, 'loss': total_loss / total, 'binary_macro_f1': score, 'seconds': time.time() - started}
        history.append(row)
        print(f"fold {fold} epoch {epoch + 1}: loss={row['loss']:.4f} binary_macro_f1={score:.4f} time={row['seconds']/60:.1f}m")
        if score > best_score + 1e-6:
            best_score, stale = score, 0
            torch.save({'model_state': ema.module.state_dict(), 'epoch': epoch + 1, 'binary_macro_f1': score}, checkpoint_path)
        else:
            stale += 1
            if stale >= PATIENCE:
                break

    best = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(best['model_state'])
    valid_loader = make_loader(MultiViewDataset(valid_all, training=False), EVAL_BATCH_SIZE)
    test_loader = make_loader(MultiViewDataset(test_frame, training=False, labeled=False), EVAL_BATCH_SIZE)
    valid_result = predict_specialist(model, valid_loader)
    test_result = predict_specialist(model, test_loader)
    np.savez_compressed(array_paths['valid'], **valid_result)
    np.savez_compressed(array_paths['test'], **test_result)
    pd.DataFrame(history).to_csv(output_fold / 'history.csv', index=False)
    metadata = {'fold': fold, 'best_epoch': int(best['epoch']), 'binary_macro_f1': float(best['binary_macro_f1'])}
    complete.write_text(json.dumps(metadata, indent=2))
    del model, ema
    torch.cuda.empty_cache()
    return valid_result, test_result, metadata

## 5. Jalankan lima specialist OOF

Setiap fold memakai checkpoint model utama fold yang sama. Dengan demikian specialist tidak pernah belajar dari validation fold yang sedang dinilai.

In [ ]:
specialist_oof = np.full(len(train_frame), np.nan, dtype=np.float32)
specialist_global_oof = np.full(len(train_frame), np.nan, dtype=np.float32)
specialist_view_max_oof = np.full(len(train_frame), np.nan, dtype=np.float32)
specialist_view_min_oof = np.full(len(train_frame), np.nan, dtype=np.float32)
specialist_test_sum = np.zeros(len(test_frame), dtype=np.float64)
fold_rows = []

for fold in range(FOLDS):
    valid_result, test_result, metadata = train_specialist_fold(fold)
    valid_indices = train_frame.index[train_frame.fold.eq(fold)].to_numpy()
    specialist_oof[valid_indices] = valid_result['probability']
    specialist_global_oof[valid_indices] = valid_result['global_probability']
    specialist_view_max_oof[valid_indices] = valid_result['view_max']
    specialist_view_min_oof[valid_indices] = valid_result['view_min']
    specialist_test_sum += test_result['probability']
    fold_rows.append(metadata)

if not all(np.isfinite(array).all() for array in [specialist_oof, specialist_global_oof, specialist_view_max_oof, specialist_view_min_oof]):
    raise RuntimeError('Specialist OOF tidak lengkap.')
specialist_test = (specialist_test_sum / FOLDS).astype(np.float32)
np.save(OUTPUT_DIR / 'specialist_oof.npy', specialist_oof)
np.save(OUTPUT_DIR / 'specialist_test.npy', specialist_test)
pd.DataFrame(fold_rows).to_csv(OUTPUT_DIR / 'specialist_fold_metrics.csv', index=False)
display(pd.DataFrame(fold_rows))

## 6. Cross-fitted fusion dengan perlindungan setiap kelas

`Lainnya` tetap berasal dari model utama. Specialist hanya membagi sisa probabilitas antara Aman dan Rusak. Bobot specialist dan bias dipilih pada empat fold lalu dinilai pada fold kelima. Kandidat primary ditolak bila Rusak tidak membaik atau Aman/Lainnya turun melewati toleransi.

In [ ]:
def logit(probability):
    probability = np.clip(probability, 1e-6, 1 - 1e-6)
    return np.log(probability) - np.log1p(-probability)


def compose_probabilities(main_probabilities, specialist_probability, alpha, rusak_bias, lainnya_bias):
    aman = main_probabilities[:, CLASS_TO_INDEX['Aman']]
    rusak = main_probabilities[:, CLASS_TO_INDEX['Rusak']]
    lainnya = main_probabilities[:, CLASS_TO_INDEX['Lainnya']]
    main_binary = rusak / np.clip(aman + rusak, 1e-7, None)
    fused_binary = 1 / (1 + np.exp(-(
        (1 - alpha) * logit(main_binary)
        + alpha * logit(specialist_probability)
        + rusak_bias
    )))
    fused_lainnya = 1 / (1 + np.exp(-(logit(lainnya) + lainnya_bias)))
    remainder = 1 - fused_lainnya
    return np.column_stack((
        remainder * (1 - fused_binary),
        remainder * fused_binary,
        fused_lainnya,
    )).astype(np.float32)


def class_f1(target_values, probabilities):
    return precision_recall_fscore_support(
        target_values, probabilities.argmax(1), labels=range(len(CLASSES)), zero_division=0
    )[2]


def choose_parameters(main_probability, specialist_probability, target_values):
    baseline_f1 = class_f1(target_values, main_probability)
    best = {'score': f1_score(target_values, main_probability.argmax(1), average='macro'),
            'alpha': 0.0, 'rusak_bias': 0.0, 'lainnya_bias': 0.0, 'class_f1': baseline_f1}
    for alpha in np.arange(0.0, 1.001, 0.10):
        for rusak_bias in np.arange(-0.40, 0.401, 0.05):
            for lainnya_bias in np.arange(-0.20, 0.201, 0.10):
                candidate = compose_probabilities(main_probability, specialist_probability, alpha, rusak_bias, lainnya_bias)
                candidate_f1 = class_f1(target_values, candidate)
                if candidate_f1[0] < baseline_f1[0] - MAX_PROTECTED_CLASS_DROP:
                    continue
                if candidate_f1[2] < baseline_f1[2] - MAX_PROTECTED_CLASS_DROP:
                    continue
                score = f1_score(target_values, candidate.argmax(1), average='macro')
                if score > best['score'] + 1e-12:
                    best = {'score': score, 'alpha': alpha, 'rusak_bias': rusak_bias,
                            'lainnya_bias': lainnya_bias, 'class_f1': candidate_f1}
    return best


crossfit_probabilities = np.zeros_like(main_oof)
parameter_rows = []
for held_out_fold in range(FOLDS):
    calibration = train_frame.fold.ne(held_out_fold).to_numpy()
    held_out = ~calibration
    selected = choose_parameters(main_oof[calibration], specialist_oof[calibration], targets[calibration])
    crossfit_probabilities[held_out] = compose_probabilities(
        main_oof[held_out], specialist_oof[held_out],
        selected['alpha'], selected['rusak_bias'], selected['lainnya_bias'],
    )
    parameter_rows.append({
        'held_out_fold': held_out_fold,
        'alpha': selected['alpha'],
        'rusak_bias': selected['rusak_bias'],
        'lainnya_bias': selected['lainnya_bias'],
        'calibration_macro_f1': selected['score'],
        'held_out_macro_f1': f1_score(targets[held_out], crossfit_probabilities[held_out].argmax(1), average='macro'),
    })

baseline_macro = f1_score(targets, main_oof.argmax(1), average='macro')
crossfit_macro = f1_score(targets, crossfit_probabilities.argmax(1), average='macro')
baseline_class_f1 = class_f1(targets, main_oof)
crossfit_class_f1 = class_f1(targets, crossfit_probabilities)
crossfit_gate_passed = (
    crossfit_macro >= baseline_macro + MIN_MACRO_GAIN
    and crossfit_class_f1[CLASS_TO_INDEX['Rusak']] >= baseline_class_f1[CLASS_TO_INDEX['Rusak']] + MIN_RUSAK_F1_GAIN
    and crossfit_class_f1[CLASS_TO_INDEX['Aman']] >= baseline_class_f1[CLASS_TO_INDEX['Aman']] - MAX_PROTECTED_CLASS_DROP
    and crossfit_class_f1[CLASS_TO_INDEX['Lainnya']] >= baseline_class_f1[CLASS_TO_INDEX['Lainnya']] - MAX_PROTECTED_CLASS_DROP
)

parameter_frame = pd.DataFrame(parameter_rows)
final_parameters = {
    key: float(parameter_frame[key].median())
    for key in ['alpha', 'rusak_bias', 'lainnya_bias']
}
candidate_oof = compose_probabilities(main_oof, specialist_oof, **final_parameters)
candidate_test = compose_probabilities(main_test, specialist_test, **final_parameters)
candidate_macro = f1_score(targets, candidate_oof.argmax(1), average='macro')
candidate_class_f1 = class_f1(targets, candidate_oof)
candidate_gate_passed = (
    candidate_macro >= baseline_macro + MIN_MACRO_GAIN
    and candidate_class_f1[CLASS_TO_INDEX['Rusak']] >= baseline_class_f1[CLASS_TO_INDEX['Rusak']] + MIN_RUSAK_F1_GAIN
    and candidate_class_f1[CLASS_TO_INDEX['Aman']] >= baseline_class_f1[CLASS_TO_INDEX['Aman']] - MAX_PROTECTED_CLASS_DROP
    and candidate_class_f1[CLASS_TO_INDEX['Lainnya']] >= baseline_class_f1[CLASS_TO_INDEX['Lainnya']] - MAX_PROTECTED_CLASS_DROP
)
accepted = crossfit_gate_passed and candidate_gate_passed
primary_oof = candidate_oof if accepted else main_oof
primary_test = candidate_test if accepted else main_test

comparison = pd.DataFrame({
    'baseline': [baseline_macro, *baseline_class_f1],
    'crossfit_specialist': [crossfit_macro, *crossfit_class_f1],
    'candidate_median_parameters': [
        candidate_macro, *candidate_class_f1
    ],
}, index=['macro_f1', *[f'f1_{label}' for label in CLASSES]])
display(parameter_frame)
display(comparison)
print('Specialist accepted for primary:', accepted)
print('Median parameters:', final_parameters)

parameter_frame.to_csv(OUTPUT_DIR / 'fusion_crossfit_parameters.csv', index=False)
comparison.to_csv(OUTPUT_DIR / 'fusion_oof_comparison.csv')
np.save(OUTPUT_DIR / 'fusion_crossfit_oof.npy', crossfit_probabilities)
np.save(OUTPUT_DIR / 'fusion_candidate_test.npy', candidate_test)

primary_submission = template.copy()
primary_submission['label'] = np.asarray(CLASSES)[primary_test.argmax(1)]
primary_submission.to_csv(OUTPUT_DIR / 'submission_rusak_specialist_primary.csv', index=False)
candidate_submission = template.copy()
candidate_submission['label'] = np.asarray(CLASSES)[candidate_test.argmax(1)]
candidate_submission.to_csv(OUTPUT_DIR / 'submission_rusak_specialist_candidate.csv', index=False)

report = pd.DataFrame(classification_report(
    targets, primary_oof.argmax(1), target_names=CLASSES, output_dict=True, zero_division=0
)).T
report.to_csv(OUTPUT_DIR / 'primary_oof_classification_report.csv')
display(report)

matrix = confusion_matrix(targets, primary_oof.argmax(1), normalize='true')
plt.figure(figsize=(6, 5))
sns.heatmap(matrix, annot=True, fmt='.3f', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Prediksi'); plt.ylabel('Label'); plt.title('Primary OOF'); plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'primary_oof_confusion_matrix.png', dpi=170); plt.show()

## 7. Kandidat perbaikan label Rusak

Tiga sinyal yang digunakan:

- conditional Aman–Rusak dari ConvNeXt utama;
- multi-view binary specialist;
- probabilitas OOF teacher lama yang tersimpan di manifest.

Kandidat tetap berstatus `pending_manual_review`. Notebook tidak melakukan hard relabel tanpa CSV keputusan manual.

In [ ]:
main_binary = main_oof[:, CLASS_TO_INDEX['Rusak']] / np.clip(
    main_oof[:, CLASS_TO_INDEX['Aman']] + main_oof[:, CLASS_TO_INDEX['Rusak']], 1e-7, None
)
official_binary = train_frame.label_original.eq('Rusak').to_numpy()
main_vote = main_binary >= 0.5
specialist_vote = specialist_oof >= 0.5
legacy_vote = legacy_binary >= 0.5
consensus_vote = main_vote & specialist_vote & legacy_vote
consensus_aman = (~main_vote) & (~specialist_vote) & (~legacy_vote)
all_agree = consensus_vote | consensus_aman

main_confidence = np.maximum(main_binary, 1 - main_binary)
specialist_confidence = np.maximum(specialist_oof, 1 - specialist_oof)
legacy_confidence = np.maximum(legacy_binary, 1 - legacy_binary)
view_consistent = np.where(
    consensus_vote,
    specialist_view_max_oof >= MIN_SPECIALIST_CONFIDENCE,
    specialist_view_max_oof <= 1 - MIN_SPECIALIST_CONFIDENCE,
)
candidate_mask = (
    train_frame.label_original.isin(['Aman', 'Rusak']).to_numpy()
    & all_agree
    & (consensus_vote != official_binary)
    & (main_confidence >= MIN_MAIN_CONFIDENCE)
    & (specialist_confidence >= MIN_SPECIALIST_CONFIDENCE)
    & (legacy_confidence >= MIN_LEGACY_CONFIDENCE)
    & view_consistent
)

review = train_frame.loc[candidate_mask, [
    'filepath', 'image_path', 'fold', 'label_original', 'pixel_sha256', 'split_group'
]].copy()
indices = np.flatnonzero(candidate_mask)
review['proposed_label'] = np.where(consensus_vote[indices], 'Rusak', 'Aman')
review['main_probability_rusak'] = main_binary[indices]
review['specialist_probability_rusak'] = specialist_oof[indices]
review['legacy_probability_rusak'] = legacy_binary[indices]
review['specialist_global_probability_rusak'] = specialist_global_oof[indices]
review['specialist_view_max_probability_rusak'] = specialist_view_max_oof[indices]
review['specialist_view_min_probability_rusak'] = specialist_view_min_oof[indices]
review['consensus_confidence'] = np.minimum.reduce([
    main_confidence[indices], specialist_confidence[indices], legacy_confidence[indices]
])
review['review_status'] = 'pending_manual_review'
review['manual_decision'] = ''
review['manual_label'] = ''
review['manual_notes'] = ''
review = review.sort_values('consensus_confidence', ascending=False).reset_index(drop=True)
review.insert(0, 'audit_id', [f'CONSENSUS-{i:04d}' for i in range(1, len(review) + 1)])
review.to_csv(OUTPUT_DIR / 'rusak_consensus_review_candidates.csv', index=False)


def save_review_contact_sheets(review_frame, page_size=20):
    sheet_paths = []
    card_width, card_height, columns = 360, 290, 4
    rows_per_page = math.ceil(page_size / columns)
    for page_number, start in enumerate(range(0, len(review_frame), page_size), start=1):
        page = review_frame.iloc[start:start + page_size]
        canvas = Image.new('RGB', (columns * card_width, rows_per_page * card_height), 'white')
        draw = ImageDraw.Draw(canvas)
        for position, row in enumerate(page.itertuples(index=False)):
            column, card_row = position % columns, position // columns
            left, top = column * card_width, card_row * card_height
            with load_rgb(row.image_path) as image:
                image.thumbnail((340, 215), Image.Resampling.LANCZOS)
                image_left = left + (card_width - image.width) // 2
                canvas.paste(image, (image_left, top + 8))
            draw.text((left + 8, top + 228), f'{row.audit_id} | {Path(row.filepath).name}', fill='black')
            draw.text((left + 8, top + 247), f'{row.label_original} -> {row.proposed_label}', fill='#9f2f25')
            draw.text((left + 8, top + 266), f'consensus min={row.consensus_confidence:.1%}', fill='#17683c')
        path = OUTPUT_DIR / f'rusak_consensus_contact_sheet_{page_number:02d}.jpg'
        canvas.save(path, quality=88, optimize=True)
        sheet_paths.append(str(path))
    return sheet_paths


contact_sheet_paths = save_review_contact_sheets(review) if len(review) else []

refined = train_frame.copy()
refined['label_refined'] = refined.label_original
refined['refinement_status'] = 'official_original'
refined['training_weight'] = 1.0

manual_path = Path(MANUAL_REVIEW_CSV).expanduser() if MANUAL_REVIEW_CSV else None
applied_manual_changes = 0
if manual_path and manual_path.is_file():
    manual = pd.read_csv(manual_path)
    needed = {'filepath', 'manual_decision', 'manual_label'}
    if not needed.issubset(manual.columns):
        raise ValueError(f'CSV review harus memiliki kolom {sorted(needed)}')
    manual = manual.drop_duplicates('filepath', keep='last').set_index('filepath')
    for index, row in refined.iterrows():
        if row.filepath not in manual.index:
            continue
        decision = str(manual.at[row.filepath, 'manual_decision']).strip()
        manual_label = str(manual.at[row.filepath, 'manual_label']).strip()
        if decision == 'relabel' and manual_label in CLASSES:
            refined.at[index, 'label_refined'] = manual_label
            refined.at[index, 'refinement_status'] = 'manual_relabel'
            applied_manual_changes += int(manual_label != row.label_original)
        elif decision == 'keep_original':
            refined.at[index, 'refinement_status'] = 'manual_keep_original'
        elif decision == 'unclear':
            refined.at[index, 'refinement_status'] = 'manual_unclear'
            refined.at[index, 'training_weight'] = 0.5

refined['label'] = refined.label_refined
refined['label_final'] = refined.label_refined
refined.to_csv(OUTPUT_DIR / 'refined_training_manifest.csv', index=False)

review_summary = {
    'candidate_count': int(len(review)),
    'candidate_transitions': {
        f'{source} -> {target}': int(count)
        for (source, target), count in review.groupby(['label_original', 'proposed_label']).size().items()
    },
    'automatic_hard_relabels': 0,
    'manual_changes_applied': int(applied_manual_changes),
    'specialist_accepted_for_primary': bool(accepted),
    'baseline_macro_f1': float(baseline_macro),
    'crossfit_specialist_macro_f1': float(crossfit_macro),
    'final_parameters': final_parameters,
    'contact_sheets': contact_sheet_paths,
}
(OUTPUT_DIR / 'run_summary.json').write_text(json.dumps(review_summary, indent=2, ensure_ascii=False, default=str))
display(review.head(30))
print(json.dumps(review_summary, indent=2, ensure_ascii=False, default=str))
print('Primary submission:', OUTPUT_DIR / 'submission_rusak_specialist_primary.csv')
print('Review candidates:', OUTPUT_DIR / 'rusak_consensus_review_candidates.csv')
print('Refined manifest:', OUTPUT_DIR / 'refined_training_manifest.csv')
print('Contact sheets:', len(contact_sheet_paths))

## Cara memakai hasil

1. Periksa tabel `fusion_oof_comparison.csv` dan baris `Specialist accepted for primary`.
2. Jika `True`, submit `submission_rusak_specialist_primary.csv`. Jika `False`, file primary identik dengan baseline dan tidak perlu disubmit ulang; `candidate` hanya untuk diagnosis.
3. Review `rusak_consensus_review_candidates.csv`. Isi `manual_decision` dengan `keep_original`, `relabel`, atau `unclear`; isi `manual_label` hanya ketika memilih `relabel`.
4. Atur `MANUAL_REVIEW_CSV`, lalu jalankan ulang sel tahap 7 untuk menghasilkan `refined_training_manifest.csv`.
5. Manifest refined menjadi input training tiga kelas berikutnya. Kandidat yang belum diperiksa tidak diubah dan tidak diturunkan bobotnya.

Tidak ada klaim bahwa konfigurasi ini pasti mencapai 0,92. Keputusan primary didasarkan pada OOF cross-fold dan perlindungan per kelas agar eksperimen tidak secara sengaja menurunkan Aman atau Lainnya.